In [ ]:
#libraries needed for pipeline
import pandas as pd
import numpy as np
import urllib
import os 
import requests
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import seaborn as seabornInstance 
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression
from sklearn import metrics
get_ipython().magic(u'matplotlib inline')
import time
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.Blast import NCBIWWW
from Bio.Blast import NCBIXML
from Bio.PDB import *
import os, shutil
from biopandas.pdb import PandasPdb
from Bio import ExPASy
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.firefox.options import Options
import pymol as p

USE THIS LINK TO FIND IF GENE HAS RELEVANT COMPOUNDS IN PUBCHEM

REPLACE Synpo with your gene of interest. And download the summary csv file if there are bioassays for this target.

https://pubchem.ncbi.nlm.nih.gov/#query=Synpo&tab=bioassay

In [ ]:
https://pubchem.ncbi.nlm.nih.gov/#query=IGFBP7&&tab=bioassay

In [ ]:
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.Seq import Seq
from Bio import SeqIO
from http.client import IncompleteRead
import time

# Option 1: paste a raw sequence
query = Seq(input("Paste protein sequence: ").strip())

# Option 2: use a FASTA file instead
# record = SeqIO.read("sample.fasta", "fasta")
# query = record.seq

def run_blast(seq, max_retries=3):
    for attempt in range(max_retries):
        try:
            print("Query sent to NCBIWWW against swissprot database")
            result_handle = NCBIWWW.qblast("blastp", "swissprot", str(seq))
            xml_text = result_handle.read()
            result_handle.close()

            with open("my_blast_result.xml", "w") as blast_result:
                blast_result.write(xml_text)

            return True

        except IncompleteRead as e:
            print(f"BLAST download interrupted: {e}")
            if attempt < max_retries - 1:
                print("Retrying...")
                time.sleep(5)
            else:
                raise

        except Exception as e:
            print(f"BLAST failed: {e}")
            raise

    return False

run_blast(query)

hit_ids = []
number_of_alignments = 0

with open("my_blast_result.xml") as result:
    blast_records = NCBIXML.parse(result)
    for rec in blast_records:
        for alignment in rec.alignments:
            hit_ids.append(alignment.accession)
            number_of_alignments += 1
            if number_of_alignments == 3:
                break
        if number_of_alignments == 3:
            break

print(f"Top hits: {hit_ids}")

if len(hit_ids) == 0:
    print("No hits identified in the database. Check your sequence input.")

with open("target", "r") as df:
    found = False
    for line in df:
        parts = line.split()
        if len(parts) < 3:
            continue
        for entry in hit_ids:
            if entry in parts[2]:
                print(f"Entry {entry} present in database!")
                check_id = entry
                found = True
                break
        if found:
            break

if not found:
    print("Target not found in database")

In [ ]:
link1 = "https://pubchem.ncbi.nlm.nih.gov/sdq/sdqagent.cgi?infmt=json&outfmt=csv&query=%7B%22download%22%3A%22*%22%2C%22collection%22%3A%22bioactivity%22%2C%22where%22%3A%7B%22ands%22%3A%5B%7B%22protacxn%22%3A%22notnull%22%7D%2C%7B%22cid%22%3A%22notnull%22%7D%2C%7B%22repacxn%22%3A%22P0C6U8%22%7D%5D%7D%2C%22order%22%3A%5B%22activity%2Casc%22%5D%2C%22start%22%3A1%2C%22limit%22%3A10000000%2C%22downloadfilename%22%3A%22%7BPROTACXN_P0C6U8%7D_bioactivity_protein%22%7D"
check_id='P13497'
link1 = link1.replace('P0C6U8', check_id)
print(link1) 

for i in range(2):
    try:
        req = requests.get(link1)
        url_content = req.content
        csv_file = open('downloaded1_example.csv', 'wb')
        csv_file.write(url_content)
        csv_file.close()
        print("Completed the Request")
        break
    #except IncompleteRead as I:
     #   print("Server Overloading , Proceeding")
      #  break
    except Exception as a:
        print(str(a)+" is the error , Trying {} time".format(i))
        continue
    else:
        break
else:
    print("something Wrong , Try running Again [refer error code for more]")
data = pd.read_csv("downloaded1_example.csv",on_bad_lines='skip')

data

In [ ]:
# --- Step 1: Load Summary CSV ---
#summary_file = "PubChem_bioassay_IGFBP7.csv"  # replace with your file
import pandas as pd
import requests
from io import StringIO

# --- Step 1: Load Summary CSV ---
summary_file = "PATH/TO/YOUR/bioassay_summary.csv  # EDIT: replace with your local path"  # replace with your file
summary_df = pd.read_csv(summary_file)


# Filter for target assays
target_aids = summary_df[summary_df['aidname'].str.contains("", case=False, na=False)]['aid'].tolist() # replace sparc with your gene
print(f"Found {len(target_aids)} target assay(s): {target_aids}")


## TESTING: Limit to 5 assays for quick testing
target_aids = target_aids[:30]
# target_aids = [target_aids[i] for i in [6,7,8,9]]# Limit to first 5 assays for testing


# -----------------------------
# Step 2: Download compounds per AID
# -----------------------------
all_compounds = []

for aid in target_aids:
    print(f"Downloading compounds for AID {aid}...")
    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{aid}/CSV'
    try:
        r = requests.get(url)
        r.raise_for_status()
        csv_data = StringIO(r.text)
        df = pd.read_csv(csv_data, on_bad_lines='skip')
        df['AID'] = aid  # Add assay ID
        all_compounds.append(df)
        print(f"Downloaded {len(df)-2} compounds for AID {aid}")
    except Exception as e:
        print(f"Failed to download AID {aid}: {e}")

# Combine all assays
if all_compounds:
    combined_df = pd.concat(all_compounds, ignore_index=True)
else:
    combined_df = pd.DataFrame()

print(f"Total rows before deduplication: {len(combined_df)}")
print("Columns in combined_df:", combined_df.columns.tolist())

# -----------------------------
# Step 3: Choose numeric activity column
# -----------------------------
# Target2DeNovoDrug expects 'acvalue', so we'll use Standard Value if present,
# otherwise fallback to PUBCHEM_ACTIVITY_SCORE
activity_col = 'Standard Value' if 'Standard Value' in combined_df.columns else 'PUBCHEM_ACTIVITY_SCORE'

# Convert to numeric
combined_df[activity_col] = pd.to_numeric(combined_df[activity_col], errors='coerce')

# -----------------------------
# Step 4: Deduplicate by compound
# -----------------------------
cid_col = 'PUBCHEM_CID'

if not combined_df.empty:
    # Aggregate multiple rows per CID: take mean activity value
    data = combined_df.groupby(cid_col, as_index=False)[activity_col].mean()
else:
    data = pd.DataFrame(columns=[cid_col, activity_col])

# -----------------------------
# Step 5: Rename columns to notebook format
# -----------------------------
data = data.rename(columns={
    cid_col: 'cid',
    activity_col: 'acvalue'
})

# Drop rows with missing activity
data = data.dropna(subset=['acvalue'])

# -----------------------------
# Step 6: Ready for Target2DeNovoDrug
# -----------------------------
print("Data ready for Target2DeNovoDrug pipeline:")
print(data.head())
print(f"Total unique compounds: {len(data)}")

In [ ]:
# Run this cell first when switching to a new target/dataset
import os
import json

# Clear the disk cache
if os.path.exists("cid_cache.json"):
    os.remove("cid_cache.json")
    print("Cache cleared")

# Reset all key variables
cache        = {}
x_rows       = []
y_rows       = []
x_data       = None
y_data       = None
train_file   = None
candidate_cids = []
all_similar_cids = []

print("All variables reset — ready for new target")

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import io
import os
import json
from threading import Lock
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed

# =========================
# VARIABLES TO CHANGE
# =========================
original_cid_limit   = 10_000_000  # max original CIDs to use
target_ok            = 1_000_000   # stop after this many accepted rows
cid_keep             = 30          # max similar CIDs per original CID
BATCH_SIZE           = 100         # CIDs per batch API call (PubChem max = 100)
MAX_WORKERS_SIMILAR  = 10          # parallel threads for similar-CID fetching
MAX_WORKERS_PROPS    = 8           # parallel threads for property batch fetching
MAX_WORKERS_ASSAY    = 25          # parallel threads for assay fetching
CACHE_FILE           = "cid_cache.json"

# =========================
# STEP 0: Session + helpers
# =========================
def build_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=6, connect=6, read=6,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"
        )
    })
    return session

session = build_session()
cache_lock = Lock()

# Load or initialise disk cache with corruption handling
if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE) as f:
            cache = json.load(f)
        print(f"Loaded cache with {len(cache)} entries")
    except (json.JSONDecodeError, ValueError):
        print("Cache file corrupted — deleting and starting fresh")
        os.remove(CACHE_FILE)
        cache = {}
else:
    cache = {}

def save_cache():
    """Atomic save — never leaves cache in a corrupted state."""
    with cache_lock:
        cache_copy = dict(cache)
    tmp = CACHE_FILE + ".tmp"
    with open(tmp, "w") as f:
        json.dump(cache_copy, f)
    os.replace(tmp, CACHE_FILE)

def fetch_text(url, session, timeout=60, retries=2, sleep_sec=3):
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, timeout=timeout)
            if r.status_code == 404:
                return None
            r.raise_for_status()
            return r.text
        except requests.exceptions.RequestException as e:
            print(f"  Attempt {attempt}/{retries} failed: {e}")
            if attempt < retries:
                time.sleep(sleep_sec)
    return None

def fetch_csv(url, session, timeout=60, retries=2, sleep_sec=3):
    text = fetch_text(url, session=session, timeout=timeout,
                      retries=retries, sleep_sec=sleep_sec)
    if not text or not text.strip():
        return None
    try:
        return pd.read_csv(io.StringIO(text))
    except Exception as e:
        print(f"  CSV parse error: {e}")
        return None

# =========================
# STEP 1: Clean input data
# =========================
new_data = data[["cid", "acvalue"]].dropna().copy()
new_data["cid"]     = pd.to_numeric(new_data["cid"],     errors="coerce")
new_data["acvalue"] = pd.to_numeric(new_data["acvalue"], errors="coerce")
new_data = new_data.dropna()
new_data = new_data[new_data["acvalue"] > 0].copy()
new_data = new_data.iloc[:original_cid_limit].copy()

cid_list  = new_data["cid"].astype(int).tolist()
cid_value = cid_list.copy()

PIC50_value = (-np.log10(new_data["acvalue"] * 1e-6)).tolist()
y_data = pd.DataFrame(PIC50_value, columns=["y"])
print(f"Original CIDs loaded: {len(cid_list)}")

sub_link_fixed = (
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/"
    "fastsimilarity_2d/cid/replaceme/cids/TXT"
)

# =========================
# STEP 2: Batch-fetch properties for original CIDs (parallel batches)
# =========================
print("\n=== STEP 2: Fetching properties for original CIDs (parallel batched) ===")

def fetch_property_batch(batch):
    """Fetch one batch of up to 100 CIDs — runs in parallel."""
    time.sleep(0.1)
    cids_str = ",".join(str(c) for c in batch)
    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cids_str}/"
        "property/MolecularWeight,HeavyAtomCount,XLOGP,Complexity,"
        "HBondAcceptorCount,MonoisotopicMass,RotatableBondCount,TPSA/CSV"
    )
    return fetch_csv(url, session=session, timeout=90, retries=3, sleep_sec=3)

x_data_rows = []
orig_batches = [cid_list[i:i+BATCH_SIZE] for i in range(0, len(cid_list), BATCH_SIZE)]

with ThreadPoolExecutor(max_workers=MAX_WORKERS_PROPS) as executor:
    futures = {executor.submit(fetch_property_batch, batch): i
               for i, batch in enumerate(orig_batches)}
    for future in as_completed(futures):
        idx = futures[future]
        df = future.result()
        if df is not None and not df.empty:
            x_data_rows.append(df)
        print(f"  Batch {idx+1}/{len(orig_batches)} done")

x_data = pd.concat(x_data_rows, ignore_index=True) if x_data_rows else pd.DataFrame()
print(f"Properties fetched for original CIDs: {len(x_data)} rows")

# =========================
# STEP 3: Parallel fetch of similar CIDs
# =========================
print("\n=== STEP 3: Fetching similar CIDs (parallel) ===")

def fetch_similar_for_one(cid):
    cache_key = f"similar_{cid}"
    with cache_lock:
        if cache_key in cache:
            return cid, cache[cache_key]
    time.sleep(0.1)
    url = sub_link_fixed.replace("replaceme", str(cid))
    txt = fetch_text(url, session=session, timeout=60, retries=2, sleep_sec=2)
    if txt is None:
        return cid, []
    try:
        parts = txt.replace("\n", ",").replace(" ", "").replace("'", "").split(",")
        candidates = [int(x) for x in parts if x.strip().isdigit()][:cid_keep]
    except Exception:
        candidates = []
    with cache_lock:
        cache[cache_key] = candidates
    return cid, candidates

all_similar_cids = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS_SIMILAR) as executor:
    futures = {executor.submit(fetch_similar_for_one, cid): cid for cid in cid_value}
    for i, future in enumerate(as_completed(futures)):
        cid, similar = future.result()
        all_similar_cids.extend(similar)
        if (i + 1) % 100 == 0:
            print(f"  Progress: {i+1}/{len(cid_value)} original CIDs processed")
            save_cache()

save_cache()
print(f"Total similar CIDs collected: {len(all_similar_cids)}")

candidate_cids = list(dict.fromkeys(cid_list + all_similar_cids))
print(f"Total unique candidate CIDs: {len(candidate_cids)}")

# =========================
# STEP 3.5: PRE-FETCH ALL CANDIDATE PROPERTIES IN PARALLEL BATCHES
# =========================
print("\n=== STEP 3.5: Pre-fetching ALL candidate properties (parallel batched) ===")

# Only fetch CIDs not already in cache
uncached_prop_cids = [
    cid for cid in candidate_cids
    if f"props_{cid}" not in cache
]
print(f"  {len(candidate_cids)} total | {len(uncached_prop_cids)} not yet cached")

prop_batches = [
    uncached_prop_cids[i:i+BATCH_SIZE]
    for i in range(0, len(uncached_prop_cids), BATCH_SIZE)
]

completed_batches = 0

def fetch_and_cache_prop_batch(batch):
    """Fetch a property batch and write results directly into cache."""
    time.sleep(0.1)
    cids_str = ",".join(str(c) for c in batch)
    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cids_str}/"
        "property/MolecularWeight,HeavyAtomCount,XLOGP,Complexity,"
        "HBondAcceptorCount,MonoisotopicMass,RotatableBondCount,TPSA/CSV"
    )
    df = fetch_csv(url, session=session, timeout=90, retries=3, sleep_sec=3)
    if df is not None and not df.empty:
        results = {}
        for _, row in df.iterrows():
            cid_val = int(row["CID"])
            results[f"props_{cid_val}"] = row.to_dict()
        with cache_lock:
            cache.update(results)
    return len(batch)

with ThreadPoolExecutor(max_workers=MAX_WORKERS_PROPS) as executor:
    futures = {executor.submit(fetch_and_cache_prop_batch, batch): i
               for i, batch in enumerate(prop_batches)}
    for future in as_completed(futures):
        idx = futures[future]
        future.result()
        completed_batches += 1
        if completed_batches % 20 == 0:
            print(f"  Property batch progress: {completed_batches}/{len(prop_batches)}")
            save_cache()

save_cache()
print(f"All candidate properties now cached.")

# Add this before Step 4 to skip CIDs unlikely to have assay data
# Only fetch assays for CIDs that also had valid property data
candidate_cids_with_props = [
    cid for cid in candidate_cids
    if f"props_{cid}" in cache
]
print(f"Reduced from {len(candidate_cids)} to {len(candidate_cids_with_props)} candidates with known properties")

# =========================
# STEP 4: Parallel fetch of assay data
# =========================
print("\n=== STEP 4: Pre-fetching assay data (parallel) ===")

def fetch_assay_for_one(cid):
    time.sleep(0.2)  # stagger to stay under PubChem rate limit
    cache_key = f"assay_{cid}"
    with cache_lock:
        if cache_key in cache:
            return cid, cache[cache_key]

    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/"
        f"cid/{cid}/assaysummary/CSV"
    )
    assay_df = fetch_csv(url, session=session, timeout=60, retries=2, sleep_sec=3)

    y_val  = np.nan
    status = "no_data"

    if assay_df is None or assay_df.empty:
        status = "request_failed"
    elif "Activity Value [uM]" not in assay_df.columns:
        status = "no_activity_column"
    else:
        vals = pd.to_numeric(
            assay_df["Activity Value [uM]"], errors="coerce"
        ).dropna()
        vals = vals[vals > 0]
        if len(vals) == 0:
            status = "no_valid_values"
        else:
            y_val  = -np.log10(float(vals.median()) * 1e-6)
            status = "ok"

    result = {"y_val": y_val, "status": status}
    with cache_lock:
        cache[cache_key] = result
    return cid, result

uncached_assay_cids = [
    cid for cid in candidate_cids
    if f"assay_{cid}" not in cache
]
print(f"  Fetching assay data for {len(uncached_assay_cids)} uncached CIDs in parallel...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS_ASSAY) as executor:
    futures = {
        executor.submit(fetch_assay_for_one, cid): cid
        for cid in uncached_assay_cids
    }
    for i, future in enumerate(as_completed(futures)):
        future.result()
        if (i + 1) % 200 == 0:
            print(f"  Progress: {i+1}/{len(uncached_assay_cids)} assays fetched")
            save_cache()

save_cache()
print("All assay data cached.")

# =========================
# STEP 5: Screen candidates from cache (zero API calls)
# =========================
print("\n=== STEP 5: Screening candidates from cache ===")

x_rows = []
y_rows = []
processed_cids = set()

for cid in candidate_cids:
    if cid_keep == 0 and cid not in cid_list:
        break
    if len(y_rows) >= target_ok:
        break
    if cid in processed_cids:
        continue
    processed_cids.add(cid)

    prop_entry  = cache.get(f"props_{cid}")
    assay_entry = cache.get(f"assay_{cid}", {"y_val": np.nan, "status": "no_data"})

    status = assay_entry.get("status", "no_data")
    y_val  = assay_entry.get("y_val", np.nan)

    if prop_entry is None:
        continue

    if status == "ok" and not np.isnan(float(y_val)):
        x_rows.append(pd.DataFrame([prop_entry]))
        y_rows.append({"y": y_val, "status": status})
    else:
        print(f"  Rejected CID {cid}: {status}")

print(f"Accepted: {len(y_rows)} | Rejected: {len(processed_cids) - len(y_rows)}")

# =========================
# STEP 6: Final outputs
# =========================
print("\n=== STEP 6: Building final dataset ===")

if len(y_rows) < target_ok:
    print(f"Note: {len(y_rows)} rows accepted out of requested {target_ok}")

x_data     = pd.concat(x_rows, ignore_index=True) if x_rows else pd.DataFrame()
y_data     = pd.DataFrame(y_rows)

train_file = x_data.copy()
train_file["y"]      = y_data["y"].values
train_file["status"] = y_data["status"].values

final_data_frame = train_file.copy()

print("Final x_data shape:     ", x_data.shape)
print("Final y_data shape:     ", y_data.shape)
print("Final train_file shape: ", train_file.shape)
print("\nSample:")
print(train_file[["CID", "y", "status"]].head(10))

assert len(x_data) == len(y_data) == len(train_file), "Length mismatch!"
print("\nAll length checks passed.")

In [ ]:
final_data_frame_reset = final_data_frame
x_data_reset = x_data
y_data_reset = y_data["y"]
print("Final dataset shape",final_data_frame_reset.shape)
print("x_data shape",x_data_reset.shape)
print("y_data shape",y_data_reset.shape)
print(final_data_frame_reset.head())
print(x_data_reset.head())
print(y_data_reset.head())

In [ ]:
#RUN TO RESET ONLY : TESTING CAUTION
final_data_frame = final_data_frame_reset
x_data = x_data_reset
y_data = y_data_reset

In [ ]:
# data preparation for QSAR model building

x_data = pd.DataFrame(x_data).copy()
y_data = pd.DataFrame(y_data).copy()

# keep CID only if it exists
if "CID" in x_data.columns:
    cid_reg_list = x_data["CID"].tolist()
else:
    cid_reg_list = None
    print("CID column not found in x_data")

# build training file
train_file = x_data.copy().reset_index(drop=True)
train_file["y"] = y_data["y"].reset_index(drop=True).values

# drop non-feature columns
train_file.drop(columns=["CID", "status"], errors="ignore", inplace=True)

# make everything numeric
train_file = train_file.astype("float64")

# fill missing values if present
train_file = train_file.ffill()

print(train_file.head())
print(train_file.columns)

# data set of molecular features and experimental inhibition value for QSAR model building
x = list(train_file.columns)
x = [col for col in x if col != "y"]

x_ = train_file[x].values
y_ = train_file["y"].values

print(train_file.describe())

# from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(
#     x_, y_, test_size=0.1, random_state=0
# )

In [ ]:
print(train_file["y"].value_counts().sort_index())
print("Unique y values:", train_file["y"].nunique())

In [ ]:
print(train_file.corr(numeric_only=True)["y"].sort_values(ascending=False))

In [ ]:
# =============================
# 1️⃣ IMPORTS
# =============================
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split 
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score as r2_score_fn
from sklearn import preprocessing

print("STEP 1 finished")

# =============================
# 2️⃣ DATA PREPARATION
# =============================
train_file = x_data.copy().reset_index(drop=True)

train_file["y"] = y_data["y"].reset_index(drop=True).values

if "status" in y_data.columns:
    train_file["status"] = y_data["status"].reset_index(drop=True).values

train_file = train_file.ffill()

final_data_frame = final_data_frame.reset_index(drop=True).ffill()

final_data_frame_cids = final_data_frame["CID"].copy()

train_file.drop('status', axis=1, errors='ignore', inplace=True)
train_file.drop('CID', axis=1, errors='ignore', inplace=True)
final_data_frame.drop('status', axis=1, errors='ignore', inplace=True)
final_data_frame.drop('CID', axis=1, errors='ignore', inplace=True)

print("Training shape:", train_file.shape)
print("Prediction shape:", final_data_frame.shape)
print("STEP 2 finished")

print("Length x_data:", len(x_data))
print("Length y_data:", len(y_data))
print("Length train_file:", len(train_file))
assert len(x_data) == len(y_data) == len(train_file)

x = [col for col in train_file.columns if col != 'y']

x_ = train_file[x].values
y_ = train_file['y'].values
x_ = x_.astype("float64")
y_ = y_.astype("float64")

# Split FIRST — before any feature selection
X_train, X_test, y_train, y_test = train_test_split(x_, y_, test_size=0.1, random_state=0)

# Build training-only dataframe for feature selection
train_only_df = pd.DataFrame(X_train, columns=x)
train_only_df["y"] = y_train

# =============================
# 3️⃣ CROSS-VALIDATED EVALUATION FUNCTION
# =============================
def evaluate_combinations_cv(comb_list, degree, train_df):
    best_r2 = -np.inf
    best_features = None
    y = train_df["y"].values
    cv = KFold(n_splits=10, shuffle=True, random_state=0)

    for comb_group in comb_list:
        for comb in comb_group:
            features = list(comb)
            if not set(features).issubset(train_df.columns):
                continue
            X = train_df[features].values
            if X.shape[1] == 0:
                continue
            model = Pipeline([
                ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
                ("scaler", StandardScaler()),
                ("lr", LinearRegression())
            ])
            scores = cross_val_score(model, X, y, cv=cv, scoring="r2")
            mean_r2 = scores.mean()
            if mean_r2 > best_r2:
                best_r2 = mean_r2
                best_features = features

    return best_r2, best_features

print("STEP 3 finished")

# =============================
# 4️⃣ COMBINATIONS + TRAIN MODELS (DEGREE 1–3)
# =============================
from itertools import combinations
comb_list = [[]]

def sub(arr, r):
    global comb_list
    for i in r:
        comb = list(combinations(arr, i))
        comb_list.append(comb)
    return comb_list

newone = sub(x, [2, 3, 4, 5, 6])
del newone[0]

model_dict = {}

for degree in [1, 2, 3]:
    r2_val, best_features = evaluate_combinations_cv(
        newone,
        degree,
        train_only_df   # feature selection on training split only
    )
    model_dict[degree] = {
        "r2": r2_val,
        "features": best_features
    }

print("STEP 4 finished")

# =============================
# 4.5️⃣ HELD-OUT TEST SET EVALUATION
# =============================
for degree in [1, 2, 3]:
    features = model_dict[degree]["features"]
    if features is None:
        model_dict[degree]["test_r2"]  = None
        model_dict[degree]["train_r2"] = None
        continue

    feature_indices = [list(x).index(f) for f in features]

    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("lr", LinearRegression())
    ])

    model.fit(X_train[:, feature_indices], y_train)
    y_pred_test  = model.predict(X_test[:, feature_indices])
    y_pred_train = model.predict(X_train[:, feature_indices])

    model_dict[degree]["test_r2"]  = r2_score_fn(y_test,  y_pred_test)
    model_dict[degree]["train_r2"] = r2_score_fn(y_train, y_pred_train)

# =============================
# SINGLE SUMMARY PRINT
# =============================
print("\n" + "="*55)
print("  MODEL EVALUATION SUMMARY")
print("="*55)
print(f"{'Degree':<8} {'CV R²':>8} {'Train R²':>10} {'Test R²':>10}  {'Features'}")
print("-"*55)
for degree in [1, 2, 3]:
    d      = model_dict[degree]
    cv     = f"{d['r2']:.4f}"
    train  = f"{d['train_r2']:.4f}" if d['train_r2'] is not None else "N/A"
    test   = f"{d['test_r2']:.4f}"  if d['test_r2']  is not None else "N/A"
    feats  = str(d['features'])
    gap    = (d['train_r2'] - d['test_r2']) if d['train_r2'] is not None else None
    flag   = " *** OVERFIT" if gap and gap > 0.15 else (" *** WORSE THAN MEAN" if d['test_r2'] and d['test_r2'] < 0 else "")
    print(f"  {degree:<6} {cv:>8} {train:>10} {test:>10}  {feats}{flag}")
print("="*55)
print("Report: Test R² (held-out 10%) — the most honest measure")
best_degree = max([1,2,3], key=lambda d: model_dict[d]["test_r2"] or -np.inf)
print(f"Best model: Degree {best_degree}  |  Test R² = {model_dict[best_degree]['test_r2']:.4f}")
print(f"Best features: {model_dict[best_degree]['features']}")
print("="*55)

# =============================
# 5️⃣ PREDICTION FUNCTION
# =============================
def predict_top_cids(final_df, train_df, features, degree, top_n=50):
    if features is None:
        return None

    X_train_all = train_df[features].values
    y_train_all = train_df["y"].values
    X_pred      = final_df[features].values
    cid_list    = final_data_frame_cids.values

    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("lr", LinearRegression())
    ])

    model.fit(X_train_all, y_train_all)
    y_pred = model.predict(X_pred)

    return (
        pd.DataFrame({"CID": cid_list, "y_pred": y_pred})
        .sort_values("y_pred", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

# =============================
# 6️⃣ PREDICT TOP 50 PER DEGREE
# =============================
top_cid_mix_df = pd.DataFrame()

for degree in [1, 2, 3]:
    features = model_dict[degree]["features"]

    if features is None:
        print(f"Skipping degree {degree} (no valid model)")
        continue

    top_df = predict_top_cids(final_data_frame, train_file, features, degree)
    top_cid_mix_df[f"Degree {degree}"] = top_df["CID"]

    print(f"\nTop 50 CIDs – Degree {degree}")
    for i, cid in enumerate(top_df["CID"], 1):
        print(f"{i:02d}: {cid}")

top_cid_mix_df.to_csv("TOP_CID_123.csv", index=False)
print("\nSTEP 6 finished")

In [ ]:
#export results to excel
import pandas as pd

# Load your CSV
df = pd.read_csv("TOP_CID_123.csv")

df = df.astype(str)

# Save as Excel file
df.to_excel("PATH/TO/YOUR/drug_screening_results.xlsx  # EDIT: replace with your local path", index=False)

In [ ]:
# Run this BEFORE the export cell to diagnose
import requests

top_cids = top_cid_mix_df["Degree 3"].head(20).tolist()
print("Top CIDs:", top_cids)

for cid in top_cids[:5]:  # check first 5 only
    cid = int(cid)
    
    # Check names
    r1 = requests.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/synonyms/JSON", timeout=10)
    print(f"\nCID {cid} — Names status: {r1.status_code}")
    if r1.status_code == 200:
        names = r1.json()["InformationList"]["Information"][0].get("Synonym", [])
        print(f"  Names: {names[:3]}")
    
    # Check ChEMBL xref
    r2 = requests.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/xrefs/RegistryID/JSON", timeout=10)
    print(f"  ChEMBL xref status: {r2.status_code}")
    if r2.status_code == 200:
        ids = r2.json()["InformationList"]["Information"][0].get("RegistryID", [])
        chembl = [i for i in ids if i.startswith("CHEMBL")]
        print(f"  ChEMBL IDs: {chembl}")

In [ ]:
#export results to csv - optimized with retries + parallelism

import pandas as pd
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# =============================
# SETTINGS
# =============================
top_n       = 20
degree_col  = "Degree 3"
output_csv  = "PATH/TO/YOUR/output.csv  # EDIT: replace with your local path
MAX_WORKERS = 8   # parallel threads (safe for PubChem + ChEMBL)

# =============================
# SESSION WITH RETRIES
# =============================
def build_session():
    s = requests.Session()
    retry = Retry(
        total=5, connect=5, read=5,
        backoff_factor=1.5,                            # waits 0, 1.5, 3, 4.5, 6s
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update({"User-Agent": "Mozilla/5.0 QSAR-Research-Pipeline"})
    return s

session = build_session()

# =============================
# HELPER FUNCTIONS (robust)
# =============================
def safe_get_json(url, timeout=20):
    """Fetch JSON with retries. Returns None on failure (not empty dict — so callers can detect)."""
    try:
        r = session.get(url, timeout=timeout)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 404:
            return {}                                   # endpoint has no data for this CID
        return None                                     # other errors treated as retryable failure
    except Exception:
        return None

def get_pubchem_names(cid):
    cid = int(cid)
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/synonyms/JSON"
    data = safe_get_json(url)
    if not data:
        return []
    try:
        info = data.get("InformationList", {}).get("Information", [])
        if info and isinstance(info, list):
            return info[0].get("Synonym", []) or []
    except Exception:
        pass
    return []

def get_chembl_ids(cid):
    cid = int(cid)
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/xrefs/RegistryID/JSON"
    data = safe_get_json(url)
    if not data:
        return []
    try:
        info = data.get("InformationList", {}).get("Information", [])
        if info and isinstance(info, list):
            ids = info[0].get("RegistryID", []) or []
            return [i for i in ids if isinstance(i, str) and i.startswith("CHEMBL")]
    except Exception:
        pass
    return []

def get_chembl_targets(chembl_id):
    url = f"https://www.ebi.ac.uk/chembl/api/data/activity.json?molecule_chembl_id={chembl_id}&limit=100"
    data = safe_get_json(url, timeout=30)              # ChEMBL is slower
    targets = set()
    if data:
        for act in data.get("activities", []) or []:
            tname = act.get("target_pref_name")
            if tname:
                targets.add(tname)
    return list(targets)

def get_chembl_diseases(chembl_id):
    url = f"https://www.ebi.ac.uk/chembl/api/data/drug_indication.json?molecule_chembl_id={chembl_id}"
    data = safe_get_json(url, timeout=30)
    diseases = set()
    if data:
        for ind in data.get("drug_indications", []) or []:
            efo = ind.get("efo_term")
            if efo:
                diseases.add(efo)
    return list(diseases)

# =============================
# PROCESS ONE CID (parallel unit of work)
# =============================
def process_cid(cid):
    cid = int(cid)
    names      = get_pubchem_names(cid)
    chembl_ids = get_chembl_ids(cid)

    all_targets  = set()
    all_diseases = set()

    # Fetch ChEMBL target/disease data in parallel per molecule
    if chembl_ids:
        with ThreadPoolExecutor(max_workers=min(4, len(chembl_ids)*2)) as pool:
            target_futures  = [pool.submit(get_chembl_targets, cid_) for cid_ in chembl_ids]
            disease_futures = [pool.submit(get_chembl_diseases, cid_) for cid_ in chembl_ids]
            for f in as_completed(target_futures):
                all_targets.update(f.result())
            for f in as_completed(disease_futures):
                all_diseases.update(f.result())

    return {
        "CID":           cid,
        "PubChem_URL":   f"https://pubchem.ncbi.nlm.nih.gov/compound/{cid}",
        "Name":          names[0] if names else "Unknown",
        "PubChem_Names": "; ".join(names[:5]),
        "ChEMBL_IDs":    "; ".join(chembl_ids) if chembl_ids else "Not in ChEMBL",
        "Targets":       "; ".join(sorted(all_targets))  if all_targets  else None,
        "Diseases":      "; ".join(sorted(all_diseases)) if all_diseases else None,
    }

# =============================
# RUN IN PARALLEL
# =============================
top_cids = top_cid_mix_df[degree_col].head(top_n).tolist()
top_cids = [int(cid) for cid in top_cids]
print(f"Processing {len(top_cids)} CIDs in parallel (max {MAX_WORKERS} at a time)...")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_cid, cid): cid for cid in top_cids}
    for i, future in enumerate(as_completed(futures), 1):
        cid = futures[future]
        try:
            result = future.result()
            results.append(result)
            # Diagnostic line: shows what actually came back
            print(f"[{i:2d}/{len(top_cids)}] CID {cid}: "
                  f"names={len(result['PubChem_Names'].split(';')) if result['PubChem_Names'] else 0}, "
                  f"chembl={result['ChEMBL_IDs'].count('CHEMBL')}, "
                  f"targets={len(result['Targets'].split(';')) if result['Targets'] else 0}, "
                  f"diseases={len(result['Diseases'].split(';')) if result['Diseases'] else 0}")
        except Exception as e:
            print(f"[{i:2d}/{len(top_cids)}] CID {cid} FAILED: {e}")
            results.append({"CID": cid, "Name": "ERROR"})

# =============================
# SAVE — preserve original order
# =============================
df_results = pd.DataFrame(results)
df_results["order"] = df_results["CID"].apply(lambda c: top_cids.index(int(c)) if int(c) in top_cids else 999)
df_results = df_results.sort_values("order").drop(columns=["order"]).reset_index(drop=True)
df_results.to_csv(output_csv, index=False)
print(f"\nSaved {len(df_results)} rows to {output_csv}")
print(df_results[["CID", "Name", "ChEMBL_IDs"]].to_string())

In [ ]:
#export results to csv from existing Excel file

import pandas as pd
import requests
import time

# =============================
# SETTINGS — CHANGE THESE
# =============================
input_excel  = "PATH/TO/YOUR/drug_screening_results.xlsx  # EDIT: replace with your local path"
output_csv   = "PATH/TO/YOUR/CID_Info.csv  # EDIT: replace with your local path"
degree_col   = "Degree 3"   # ← change to "Degree 2" or "Degree 3" as needed
top_n        = 20           # ← how many top CIDs to process

# =============================
# LOAD CIDs FROM EXCEL
# =============================
df_input = pd.read_excel(input_excel)
print(f"Columns available: {df_input.columns.tolist()}")
print(f"Using: {degree_col}")

top_cids = df_input[degree_col].dropna().head(top_n).tolist()
top_cids = [int(float(cid)) for cid in top_cids]  # handles float CIDs like 24873735.0
print(f"Top {len(top_cids)} CIDs loaded: {top_cids[:5]}...")

# =============================
# HELPER FUNCTIONS
# =============================
def get_pubchem_names(cid):
    cid = int(cid)
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/synonyms/JSON"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            return r.json()["InformationList"]["Information"][0].get("Synonym", [])
    except:
        pass
    return []

def get_chembl_ids(cid):
    cid = int(cid)
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/xrefs/RegistryID/JSON"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            ids = r.json()["InformationList"]["Information"][0].get("RegistryID", [])
            return [i for i in ids if i.startswith("CHEMBL")]
    except:
        pass
    return []

def get_chembl_targets(chembl_id):
    url = f"https://www.ebi.ac.uk/chembl/api/data/activity.json?molecule_chembl_id={chembl_id}&limit=100"
    targets = set()
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            for act in r.json().get("activities", []):
                tname = act.get("target_pref_name")
                if tname:
                    targets.add(tname)
    except:
        pass
    return list(targets)

def get_chembl_diseases(chembl_id):
    url = f"https://www.ebi.ac.uk/chembl/api/data/drug_indication.json?molecule_chembl_id={chembl_id}"
    diseases = set()
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            for ind in r.json().get("drug_indications", []):
                efo = ind.get("efo_term")
                if efo:
                    diseases.add(efo)
    except:
        pass
    return list(diseases)

# =============================
# PROCESS TOP CIDs
# =============================
results = []

for idx, cid in enumerate(top_cids, 1):
    print(f"Processing {idx}/{len(top_cids)}: CID {cid}")

    names      = get_pubchem_names(cid)
    chembl_ids = get_chembl_ids(cid)

    all_targets  = set()
    all_diseases = set()
    for chembl_id in chembl_ids:
        all_targets.update(get_chembl_targets(chembl_id))
        all_diseases.update(get_chembl_diseases(chembl_id))
        time.sleep(0.2)

    results.append({
        "CID":           cid,
        "PubChem_URL":   f"https://pubchem.ncbi.nlm.nih.gov/compound/{cid}",
        "Name":          names[0] if names else "Unknown",
        "All_Names":     "; ".join(names[:5]),
        "ChEMBL_IDs":    "; ".join(chembl_ids) if chembl_ids else "Not in ChEMBL",
        "Targets":       "; ".join(all_targets)  if all_targets  else None,
        "Diseases":      "; ".join(all_diseases) if all_diseases else None,
        "Degree_Used":   degree_col,   # records which degree was used
    })

    time.sleep(0.1)

# =============================
# SAVE
# =============================
df_results = pd.DataFrame(results)
df_results.to_csv(output_csv, index=False)
print(f"\nDone. Saved {len(df_results)} rows to {output_csv}")
print(df_results[["CID", "Name", "ChEMBL_IDs", "Targets"]].to_string())

In [ ]:
# viewing the QSAR file with CID, names, targets and diseases for top 20 compounds (BY YIKAI YANG)
QSAR_file = "QSAR_CID_Info.csv"  # replace with your file
QSAR_df = pd.read_csv(QSAR_file)

In [ ]:
QSAR_df

Other Analysis: Automated In Silico modelling for drug leads generated from PubChem

In [ ]:
class ResSelect(Select):
    def accept_residue(self, residue):
        if is_aa(residue):
            return 1
        else:
            return 0


class ProteinPreparer:



    def __init__(self, protein: str, dir: str):
        self.protein = protein
        self.tmpdir = dir
        self.filename = self.tmpdir + '/pdb{}.ent'.format(self.protein)

    def prepare_protein(self):
        parser = PDBParser()
        io = PDBIO()
        PDBList().retrieve_pdb_file(pdb_code=self.protein, pdir=self.tmpdir,
                                    file_format='pdb')  # saves pdb in a form of ent file
        ipdb = parser.get_structure('ipdb', self.filename)  # Input pdb as a self.filename
        io.set_structure(ipdb)  # Setting structure for input pdb
        io.save(self.tmpdir + '/'+self.protein + '.pdb', ResSelect(),
                preserve_atom_numbering=True)  # saves the cleaned pdb
        os.system(prepare_protein_path + self.tmpdir + '/{}.pdb'.format(
            self.protein))  # prepares the structure by adding polar hydrogens and adding gesteiger charges
        shutil.move(self.protein + '.pdbqt', self.tmpdir)

prepare_ligand_path = 'mgltools_x86_64Linux2_1.5.6/bin/pythonsh MGLTools-1.5.6/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_ligand4.py -A bonds_hydrogens -U nphs_lps -l'

class LigandPreparer:

    def __init__(self, ligand_file: str, dir: str):
        self.ligand = ligand_file
        self.tmpdir = dir

    def prepare_ligand(self):
        url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{}/SDF'.format(self.ligand)
        urllib.request.urlretrieve(url, self.tmpdir + '/{}'.format(self.ligand + '.sdf')) # downloads the file
        os.system('obabel {} -O {} --gen3d'.format(
            self.tmpdir + '/' + self.ligand + '.sdf', self.tmpdir + '/' + self.ligand + 'prep.pdb')
        ) # generates 3d coords
        os.system(
            'obminimize -ff GAFF {} > {}'.format(
                self.tmpdir + '/' + self.ligand + 'prep.pdb', self.tmpdir + '/' + self.ligand + '.pdb'
                                                 )
        ) # minimizes using GAFF
        os.system(prepare_ligand_path + '{}'.format(self.tmpdir + '/' + self.ligand + '.pdb'
                                                    )) # adds charges, sets rotatable bonds
        shutil.move(self.ligand + '.pdbqt', self.tmpdir)

class VinaDocker:

    def __init__(self, ligentry: str, protentry: str, protein_pdbqt: str, ligand_pdbqt: str, dir: str):
        self.protein = protein_pdbqt + '.pdbqt'
        self.protpdb = protentry + '.pdb'
        self.protname = os.path.basename(self.protein)
        self.ligand = ligand_pdbqt + '.pdbqt'
        self.ligname = os.path.basename(self.ligand)
        self.tmpdir = dir
        self.docklog = './results/' + protentry + '_' + ligentry + '_docking.log' # !!
        self.dockfile =  './results/' + protentry + '_' + ligentry + '.pdbqt' # !!
        self.complex_name = protentry + '_' + ligentry + '_cplx.pdb'

    def dock_merge_plip(self):
        df = PandasPdb().read_pdb(self.tmpdir + '/' + self.protpdb)  # opens protein to calculate grid
        minx = df.df['ATOM']['x_coord'].min()
        maxx = df.df['ATOM']['x_coord'].max()
        cent_x = round((maxx + minx) / 2, 2)
        size_x = round(abs(maxx - minx) + 3, 2)
        miny = df.df['ATOM']['y_coord'].min()
        maxy = df.df['ATOM']['y_coord'].max()
        cent_y = round((maxy + miny) / 2, 2)
        size_y = round(abs(maxy - miny) + 3, 2)
        minz = df.df['ATOM']['z_coord'].min()
        maxz = df.df['ATOM']['z_coord'].max()
        cent_z = round((maxz + minz) / 2, 2)
        size_z = round(abs(maxz - minz) + 3, 2)
        assert (type(cent_x) != None), "Protein structure is damaged"
        assert (type(cent_y) != None), "Protein structure is damaged"
        assert (type(cent_z) != None), "Protein structure is damaged"


        print("Center point of docking grid for {} is as follows: x: {}, y: {}, z: {}".format(self.protein, cent_x, cent_y, cent_z))
        print("Sizes of docking grid are as follows: x: {}, y: {}, z: {}".format(size_x, size_y, size_z))
        print("Receptor: ", self.protein, "\n", "Ligand: ", self.ligand, "\n", "Output file: ", self.dockfile)
        os.system(
            'vina --receptor {} --ligand "{}" --center_x {} --center_y {} --center_z {} --size_x {} --size_y {} --size_z {} --log "{}" --out "{}"'.format(
                self.protein,
                self.ligand,
                cent_x,
                cent_y,
                cent_z,
                size_x,
                size_y,
                size_z,
                self.docklog,
                self.dockfile
            ))
        """Postprocessing of docking files"""
        # time.sleep(60)
        df.df['ATOM']['segment_id'].replace(r'.{1,}', '', regex=True, inplace=True) # Clean pdbqt inheritance
        df.df['ATOM']['blank_4'].replace(r'.{1,}', '', regex=True, inplace=True)
        docking_output_df = PandasPdb().read_pdb(self.dockfile)
        docking_output_df.df['HETATM'].drop_duplicates(subset='atom_number', keep='first', inplace=True)  # extract first model
        docking_output_df.df['HETATM']['segment_id'].replace(r'.{1,}', '', regex=True, inplace=True)
        docking_output_df.df['HETATM']['element_symbol'].replace('A', 'C', inplace=True) # OpenBabel fix
        docking_output_df.df['HETATM']['blank_4'].replace(r'.{1,}', '', regex=True, inplace=True)  # clean pdbqt inheritance
        df.df['ATOM'] = df.df['ATOM'].append(docking_output_df.df['HETATM'], ignore_index=True) # merges the files
        df.to_pdb(path = self.complex_name,
                  records=['ATOM','HETATM'],
                  gz = False,
                  append_newline=True)
        shutil.move(self.complex_name, './results')

In [ ]:
# REWRITEN CODE from CHATGPT to run AutoDock Vina using PDB file from computer (BY: YIKAI YANG)
from Bio import SwissProt
import tempfile
import os
import shutil

# SELECT YOUR CID:
top_cid = [146478919]
top_cid = [str(x) for x in top_cid]

# YOUR LOCAL PROCESSED PDB FILE
local_pdb_path = "PATH/TO/YOUR/protein_energy_minimized.pdb  # EDIT: replace with your local path"

with tempfile.TemporaryDirectory(dir=os.getcwd()) as tmpdir:

    # Part 1 - copy your processed protein structure into the temp folder
    protein_name = os.path.splitext(os.path.basename(local_pdb_path))[0]
    local_pdb_copy = os.path.join(tmpdir, protein_name + ".pdb")
    shutil.copy(local_pdb_path, local_pdb_copy)

    # Part 2 - prepare receptor pdbqt from your local pdb
    try:
        os.system(prepare_protein_path + f'"{local_pdb_copy}"')
    except Exception as e:
        print("Protein preparation error:", e)

    # Part 3 - prepare ligands
    for CID_entry in top_cid:
        try:
            LigandPreparer(CID_entry, tmpdir).prepare_ligand()
        except Exception as e:
            print("Ligand preparation error:", e)

        # Part 4 - dock the ligand
        try:
            receptor_pdbqt = os.path.join(tmpdir, protein_name + ".pdbqt")

            VinaDocker(
                protentry=protein_name,
                ligentry=CID_entry,
                protein_pdbqt=os.path.join(tmpdir, protein_name),
                ligand_pdbqt=os.path.join(tmpdir, CID_entry),
                dir=tmpdir
            ).dock_merge_plip()

        except Exception as e:
            print("Docking error:", e)

print("****Docking procedure finished****")